# USDOT Intersection Safety Challenge — Stage-1B
## Non-GT File Deep Inspection

**Purpose:** Understand every non-GT file type in the validation dataset before building an EDA pipeline.  
**Covers:** Run-level timing · Camera frame-timing · V2X CSV · Radar JSON · Cross-file timestamp alignment  
**Constraint:** No video/pcap decoding. Lightweight, structure-only inspection.

---
## 0. Setup & Config

> **Colab note:** The dataset zip (~30 GB) is downloaded directly into Colab's  
> ephemeral `/content/` disk — no Google Drive needed.  
> Heavy files (`.pcap`, `.mp4`) are **skipped** during extraction so only  
> the lightweight CSVs and JSONs are unpacked (~248 MB).  
> If you run both notebooks in the **same Colab session**, the zip and extracted  
> folder already exist — all download/extract cells detect this and skip automatically.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────
import os, re, sys, json, warnings, textwrap
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 130)
pd.set_option('display.max_colwidth', 40)

print('Imports OK.')

In [ ]:
# ── Step 1: Download dataset zip into Colab /content/ ─────────────────────
# No Google Drive required. Colab gives ~100 GB ephemeral disk.
# Skip download if the zip is already present (safe to re-run).

import os
from pathlib import Path

DOWNLOAD_URL = 'https://data.transportation.gov/download/j58q-ymi4/application%2Fx-zip-compressed'
ZIP_PATH     = '/content/isc_stage1b.zip'

zip_path = Path(ZIP_PATH)
if zip_path.exists():
    size_gb = zip_path.stat().st_size / 1e9
    print(f'Zip already present ({size_gb:.2f} GB) — skipping download.')
    print('(Running both notebooks in the same session reuses this file.)')
else:
    print('Downloading dataset (~30 GB). This takes several minutes on Colab...')
    ret = os.system(f'wget -q --show-progress -O "{ZIP_PATH}" "{DOWNLOAD_URL}"')
    if ret == 0:
        size_gb = Path(ZIP_PATH).stat().st_size / 1e9
        print(f'Download complete: {ZIP_PATH}  ({size_gb:.2f} GB)')
    else:
        raise RuntimeError('wget failed — check your internet connection and re-run.')

In [ ]:
# ── Step 2: Selective extraction — skip .pcap and video files ─────────────
# Extracts only CSVs, JSONs, and other small metadata files (~248 MB).
# Skips .pcap (LiDAR ~18 GB) and .mp4 (video ~14 GB).
# Skip extraction entirely if the folder already exists (same session).

import zipfile
import re
from pathlib import Path
from collections import Counter

ZIP_PATH     = '/content/isc_stage1b.zip'
EXTRACT_ROOT = Path('/content/isc_stage1b')

SKIP_EXTENSIONS = {'.pcap', '.mp4', '.avi', '.mkv', '.mov', '.bag', '.bin'}

if EXTRACT_ROOT.exists() and any(EXTRACT_ROOT.iterdir()):
    n_items = sum(1 for _ in EXTRACT_ROOT.rglob('*'))
    print(f'Extraction folder already exists ({n_items:,} items) — skipping.')
    print(f'Path: {EXTRACT_ROOT}')
else:
    print(f'Opening zip: {ZIP_PATH}')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        all_entries = zf.namelist()
        info_map    = {i.filename: i for i in zf.infolist()}

    # Separate extract vs skip
    to_extract = [e for e in all_entries
                  if not e.endswith('/') and Path(e).suffix.lower() not in SKIP_EXTENSIONS]
    skipped    = [e for e in all_entries
                  if not e.endswith('/') and Path(e).suffix.lower() in SKIP_EXTENSIONS]

    ext_mb  = sum(info_map[e].file_size for e in to_extract if e in info_map) / 1e6
    skip_gb = sum(info_map[e].file_size for e in skipped    if e in info_map) / 1e9
    skip_ct = Counter(Path(e).suffix.lower() for e in skipped)

    print(f'Files to extract : {len(to_extract):,}  (~{ext_mb:.0f} MB)')
    print(f'Files skipped    : {len(skipped):,}  (~{skip_gb:.1f} GB)  {dict(skip_ct)}')
    print()

    EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        for i, entry in enumerate(to_extract):
            zf.extract(entry, path=str(EXTRACT_ROOT))
            if (i + 1) % 200 == 0 or (i + 1) == len(to_extract):
                print(f'  {i+1:,} / {len(to_extract):,} extracted...', end='\r')

    print(f'\nDone. Extracted to: {EXTRACT_ROOT}')

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────
# DATA_ROOT: path to the extracted dataset root (Run_* folders live here).
# This matches what notebook 00 extracted to.
DATA_ROOT = '/content/isc_stage1b'   # <-- update if different
MAX_RUNS  = 3

root = Path(DATA_ROOT)
if not root.exists():
    raise FileNotFoundError(
        f'DATA_ROOT not found: {root}\n'
        'Run notebook 00 first, or set DATA_ROOT to your extraction path.'
    )

# ── Discover runs ─────────────────────────────────────────────────────────
run_dirs = sorted(
    [p for p in root.iterdir() if p.is_dir() and re.match(r'run_?\d+', p.name, re.IGNORECASE)],
    key=lambda p: p.name
)
selected_runs = run_dirs[:MAX_RUNS]

print(f'DATA_ROOT    : {DATA_ROOT}')
print(f'Total runs   : {len(run_dirs)}')
print(f'Selected runs: {[r.name for r in selected_runs]}')

In [ ]:
# ── Shared helper functions (used throughout the notebook) ────────────────

def find_files(run_dir: Path, patterns: list) -> list:
    """Return all files in run_dir whose names match any regex pattern."""
    results = []
    for fpath in sorted(run_dir.iterdir()):
        if fpath.is_file():
            if any(re.search(p, fpath.name, re.IGNORECASE) for p in patterns):
                results.append(fpath)
    return results


def safe_read_csv(path: Path, nrows=None) -> pd.DataFrame | None:
    """Read a CSV, trying comma then auto-detect separator."""
    try:
        return pd.read_csv(path, nrows=nrows, low_memory=False)
    except Exception:
        try:
            return pd.read_csv(path, sep=None, engine='python', nrows=nrows, low_memory=False)
        except Exception as e:
            print(f'    [WARN] Could not read {path.name}: {e}')
            return None


def parse_isc_timestamp(ts_str: str) -> float:
    """Parse ISC camera timestamp: YYYY-MM-DD-HH-MM-SS_microseconds → Unix float."""
    try:
        main, us = str(ts_str).rsplit('_', 1)
        dt = pd.to_datetime(main, format='%Y-%m-%d-%H-%M-%S')
        return dt.timestamp() + int(us) / 1_000_000
    except Exception:
        return float('nan')


def detect_ts_cols(df: pd.DataFrame) -> list:
    """Return all columns that look like timestamps."""
    return [c for c in df.columns
            if re.search(r'time|stamp|ts\b|epoch|date', c, re.IGNORECASE)]


def fmt_size(path: Path) -> str:
    sz = path.stat().st_size
    return f'{sz/1e6:.2f} MB' if sz >= 1_000_000 else f'{sz/1e3:.1f} KB' if sz >= 1000 else f'{sz} B'


def section(title: str, width: int = 60):
    print(f'\n  ── {title} {"-" * max(0, width - len(title) - 5)}')


def run_header(run_name: str, label: str = ''):
    tag = f'  {label}' if label else ''
    print(f'\n{"="*65}')
    print(f'  RUN: {run_name}{tag}')
    print(f'{"="*65}')


def missing_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Return a small DataFrame with missing-value counts and %."""
    ms = df.isnull().sum()
    ms = ms[ms > 0]
    if ms.empty:
        return pd.DataFrame(columns=['column', 'missing', '%'])
    pct = (ms / len(df) * 100).round(1)
    return pd.DataFrame({'column': ms.index, 'missing': ms.values, '%': pct.values})


print('Helper functions defined.')

---
## A. Run-Level Timing Files

Files like `ISC_Run_XXXX_ISC_all_timing.csv` and `ISC_Run_XXXX_v2xhub_timing.csv`.  
These are small (~2 KB) and likely contain synchronisation metadata across all sensors for one run.

In [ ]:
# ── A1. Detect run-level timing files ─────────────────────────────────────
# Pattern: starts with 'ISC_Run_' and ends with .csv
ISC_TIMING_PATTERN = [r'^ISC_Run_\d+.*\.csv$']

print('Run-level timing files found:\n')
isc_timing_index = {}   # run_name → list of paths

for run in selected_runs:
    files = find_files(run, ISC_TIMING_PATTERN)
    isc_timing_index[run.name] = files
    if files:
        for f in files:
            print(f'  {run.name:12s}  {f.name:55s}  {fmt_size(f)}')
    else:
        print(f'  {run.name:12s}  [none found]')

In [ ]:
# ── A2. Load and inspect each run-level timing file ───────────────────────
isc_timing_data = {}   # run_name → {filename: {'df': ..., 'ts_range': ...}}

for run in selected_runs:
    run_header(run.name, '— Run-Level Timing')
    files = isc_timing_index.get(run.name, [])

    if not files:
        print('  [MISSING] No ISC timing files in this run.')
        continue

    isc_timing_data[run.name] = {}

    for fpath in files:
        print(f'\n  File: {fpath.name}  ({fmt_size(fpath)})')
        df = safe_read_csv(fpath)
        if df is None:
            continue

        isc_timing_data[run.name][fpath.name] = {'df': df, 'path': fpath}

        section('Shape & Columns')
        print(f'    {df.shape[0]} rows × {df.shape[1]} cols')
        print(f'    {list(df.columns)}')

        section('First 10 rows')
        print(df.head(10).to_string(index=False))

        section('Data types')
        print(df.dtypes.to_string())

        section('Missing values')
        ms = missing_summary(df)
        print('    (none)' if ms.empty else ms.to_string(index=False))

        # ── Timestamp detection & parsing ─────────────────────────────────
        ts_cols = detect_ts_cols(df)
        section(f'Timestamp columns detected: {ts_cols}')
        for tc in ts_cols:
            col = df[tc]
            # Try numeric (Unix epoch)
            numeric = pd.to_numeric(col, errors='coerce')
            if numeric.notna().any():
                t_min, t_max = numeric.min(), numeric.max()
                # Heuristic: values > 1e9 → Unix seconds
                if t_min > 1e9:
                    print(f'    [{tc}]  Unix epoch — min: {t_min:.3f}  max: {t_max:.3f}')
                    print(f'           → {pd.to_datetime(t_min, unit="s")}  to  {pd.to_datetime(t_max, unit="s")}')
                else:
                    print(f'    [{tc}]  numeric — min: {t_min}  max: {t_max}')
                isc_timing_data[run.name][fpath.name]['ts_col']    = tc
                isc_timing_data[run.name][fpath.name]['ts_series'] = numeric.dropna()
            else:
                # Try ISC string format or pandas auto-parse
                parsed = pd.to_datetime(col, errors='coerce')
                valid  = parsed.dropna()
                if not valid.empty:
                    print(f'    [{tc}]  datetime string — min: {valid.min()}  max: {valid.max()}')
                else:
                    # ISC camera format
                    isc_parsed = col.astype(str).apply(parse_isc_timestamp)
                    valid2 = isc_parsed.dropna()
                    if valid2.notna().any() and not np.isnan(valid2.iloc[0]):
                        print(f'    [{tc}]  ISC format — min: {valid2.min():.3f}  max: {valid2.max():.3f}')
                    else:
                        print(f'    [{tc}]  unrecognised format — sample: {col.dropna().iloc[0] if not col.dropna().empty else "(empty)"}')

        # ── Column meaning inference ───────────────────────────────────────
        section('Column meaning inference')
        for col in df.columns:
            c_low  = col.lower()
            sample = df[col].dropna().unique()[:3].tolist()
            if re.search(r'time|stamp|epoch|date', c_low):
                tag = '⏱ timestamp'
            elif re.search(r'sensor|camera|device|source|id|name', c_low):
                tag = '🔖 identifier'
            elif re.search(r'offset|delay|latency|diff|delta', c_low):
                tag = '⏩ offset/delta'
            elif re.search(r'status|state|mode|flag|active', c_low):
                tag = '🚦 status/flag'
            elif re.search(r'count|num|total|freq', c_low):
                tag = '🔢 count'
            else:
                tag = '❓ unknown'
            print(f'    {col:<35s}  {tag:<20s}  sample: {sample}')

---
## B. Camera Frame-Timing Files

Files like `VisualCamera1_Run_XXXX_frame-timing.csv` and `ThermalCamera3_Run_XXXX_frame-timing.csv`.  
Each CSV has `Image_number` (0-based frame index into the corresponding `.mp4`) and a `Timestamp`  
in the format `YYYY-MM-DD-HH-MM-SS_microseconds`.  
There are up to 8 visual cameras and 5 thermal cameras per run.

In [ ]:
# ── B1. Detect and group camera frame-timing files ─────────────────────────
VISUAL_PATTERN  = [r'^VisualCamera\d+.*frame-timing\.csv$']
THERMAL_PATTERN = [r'^ThermalCamera\d+.*frame-timing\.csv$']

cam_index = {}   # run_name → {'visual': [...], 'thermal': [...]}

print(f'Camera frame-timing files per run:\n')
print(f'  {"Run":<12}  {"Visual cams":>12}  {"Thermal cams":>13}')
print('  ' + '─' * 42)

for run in selected_runs:
    vis  = find_files(run, VISUAL_PATTERN)
    thm  = find_files(run, THERMAL_PATTERN)
    cam_index[run.name] = {'visual': vis, 'thermal': thm}
    print(f'  {run.name:<12}  {len(vis):>12}  {len(thm):>13}')

print()
# Detailed file list for the first run
first_run = selected_runs[0]
print(f'File list for {first_run.name}:')
for kind, files in cam_index[first_run.name].items():
    for f in files:
        print(f'  [{kind:7s}]  {f.name:55s}  {fmt_size(f)}')

In [ ]:
# ── B2. Inspect one Visual and one Thermal timing file per run ─────────────
# We sample the first file from each camera type to avoid repetition.

cam_timing_data = {}   # run_name → {'visual': {...}, 'thermal': {...}}

def inspect_frame_timing(fpath: Path, label: str) -> dict | None:
    """Load a frame-timing CSV, print full inspection, return summary dict."""
    print(f'\n  File : {fpath.name}  ({fmt_size(fpath)})')

    df = safe_read_csv(fpath)
    if df is None:
        return None

    section('Columns & shape')
    print(f'    {df.shape[0]} rows × {df.shape[1]} cols')
    print(f'    {list(df.columns)}')

    section('First 10 rows')
    print(df.head(10).to_string(index=False))

    section('Missing values')
    ms = missing_summary(df)
    print('    (none)' if ms.empty else ms.to_string(index=False))

    # ── Image_number continuity check ─────────────────────────────────────
    result = {'df': df, 'path': fpath, 'label': label}

    if 'Image_number' in df.columns:
        nums = pd.to_numeric(df['Image_number'], errors='coerce').dropna().astype(int)
        gaps = nums.diff().dropna()
        non_unit = (gaps != 1).sum()
        section('Image_number')
        print(f'    range   : {nums.min()} → {nums.max()}  ({len(nums)} frames)')
        print(f'    non-consecutive gaps: {non_unit}')

    # ── Timestamp parsing ─────────────────────────────────────────────────
    ts_col = next((c for c in df.columns if re.search(r'time|stamp', c, re.IGNORECASE)), None)
    if ts_col:
        raw_sample = df[ts_col].iloc[0] if not df.empty else '(empty)'
        # Try ISC format first
        ts_unix = df[ts_col].astype(str).apply(parse_isc_timestamp)
        valid   = ts_unix.dropna()
        valid   = valid[~np.isnan(valid)]

        section(f'Timestamp [{ts_col}]  format: {raw_sample}')
        if not valid.empty:
            t_min, t_max = valid.min(), valid.max()
            duration = t_max - t_min
            n        = len(valid)
            fps      = round(n / duration, 2) if duration > 0 else float('nan')
            interval = round(duration / (n - 1), 4) if n > 1 else float('nan')

            print(f'    format   : ISC (YYYY-MM-DD-HH-MM-SS_us)')
            print(f'    min      : {pd.to_datetime(t_min, unit="s")}')
            print(f'    max      : {pd.to_datetime(t_max, unit="s")}')
            print(f'    duration : {duration:.2f} s')
            print(f'    frames   : {n}')
            print(f'    approx   : {fps} fps  (interval ≈ {interval} s)')

            result.update({'ts_unix': ts_unix, 't_min': t_min, 't_max': t_max,
                           'fps': fps, 'n_frames': n, 'ts_col': ts_col})
        else:
            # Fallback: try pandas auto-parse
            parsed = pd.to_datetime(df[ts_col], errors='coerce').dropna()
            if not parsed.empty:
                print(f'    pandas-parsed — min: {parsed.min()}  max: {parsed.max()}')
            else:
                print(f'    Could not parse timestamp. Raw sample: {raw_sample}')

    return result


for run in selected_runs:
    run_header(run.name, '— Camera Frame-Timing')
    cam_timing_data[run.name] = {}

    for kind, files in cam_index[run.name].items():
        if not files:
            print(f'  [{kind}] No files found.')
            continue

        # Inspect first file for each camera type
        sample_file = files[0]
        print(f'\n  ▶  {kind.upper()} CAMERA — inspecting: {sample_file.name}')
        result = inspect_frame_timing(sample_file, kind)
        if result:
            cam_timing_data[run.name][kind] = result

In [ ]:
# ── B3. Schema consistency across all camera files in a run ────────────────
# Do all VisualCamera timing files have the same columns?
# Do Thermal files match Visual files?

for run in selected_runs:
    run_header(run.name, '— Camera Schema Consistency')
    all_files = cam_index[run.name]['visual'] + cam_index[run.name]['thermal']

    schemas = {}
    for fpath in all_files:
        df = safe_read_csv(fpath, nrows=1)
        if df is not None:
            schemas[fpath.name] = list(df.columns)

    if not schemas:
        print('  No files loaded.')
        continue

    ref_name, ref_cols = next(iter(schemas.items()))
    all_same = all(cols == ref_cols for cols in schemas.values())

    print(f'  Reference schema ({ref_name}): {ref_cols}')
    print(f'  All {len(schemas)} files share this schema: {"YES ✓" if all_same else "NO — differences below"}')

    if not all_same:
        for fname, cols in schemas.items():
            if cols != ref_cols:
                only_ref   = set(ref_cols) - set(cols)
                only_other = set(cols) - set(ref_cols)
                print(f'  {fname}: only_ref={sorted(only_ref)}  only_other={sorted(only_other)}')

    # ── Per-camera row count summary ──────────────────────────────────────
    print(f'\n  Per-camera frame counts:')
    for kind in ['visual', 'thermal']:
        for fpath in cam_index[run.name][kind]:
            df = safe_read_csv(fpath, nrows=None)
            n  = len(df) if df is not None else '?'
            cam_label = fpath.name.split('_Run_')[0]
            print(f'    {cam_label:<25s}  {n:>6} frames')

---
## C. V2X CSV Files

Files like `V2XHubSensor_Run_XXXX.csv`.  
V2X (Vehicle-to-Everything) hub data may contain signal phase/state messages (SPaT),  
BSM messages, or intersection state data broadcast over DSRC/C-V2X.

In [ ]:
# ── C1. Detect V2X CSV files ───────────────────────────────────────────────
V2X_PATTERN = [r'v2x', r'v2xhub', r'bsm', r'spat']

print('V2X files found:\n')
v2x_index = {}

for run in selected_runs:
    files = find_files(run, V2X_PATTERN)
    v2x_index[run.name] = files
    if files:
        for f in files:
            print(f'  {run.name:<12}  {f.name:55s}  {fmt_size(f)}')
    else:
        print(f'  {run.name:<12}  [none found]')

In [ ]:
# ── C2. Inspect V2X CSV files ──────────────────────────────────────────────
v2x_data = {}

for run in selected_runs:
    files = v2x_index.get(run.name, [])
    if not files:
        continue

    run_header(run.name, '— V2X CSV')
    v2x_data[run.name] = {}

    for fpath in files:
        print(f'\n  File: {fpath.name}  ({fmt_size(fpath)})')
        df = safe_read_csv(fpath)
        if df is None:
            continue

        v2x_data[run.name][fpath.name] = {'df': df, 'path': fpath}

        section('Shape & Columns')
        print(f'    {df.shape[0]} rows × {df.shape[1]} cols')
        print(f'    {list(df.columns)}')

        section('First 10 rows')
        print(df.head(10).to_string(index=False))

        section('Missing values')
        ms = missing_summary(df)
        print('    (none)' if ms.empty else ms.to_string(index=False))

        section('Low-cardinality columns (unique values ≤ 20)')
        for col in df.columns:
            n_unique = df[col].nunique(dropna=False)
            if n_unique <= 20:
                vals = sorted(df[col].dropna().unique().tolist())
                print(f'    {col:<35s}  ({n_unique} unique): {vals}')

        section('Content type inference')
        col_names = ' '.join(df.columns).lower()
        flags = {
            'timestamps'      : bool(re.search(r'time|stamp|epoch', col_names)),
            'signal/event'    : bool(re.search(r'signal|phase|event|spat|state|light', col_names)),
            'sensor status'   : bool(re.search(r'status|active|online|mode|health', col_names)),
            'object tracking' : bool(re.search(r'track|id|object|detect|vehicle|speed', col_names)),
            'BSM/V2V data'    : bool(re.search(r'bsm|dsrc|cv2x|lat|lon|heading|speed', col_names)),
        }
        for flag, val in flags.items():
            print(f'    {flag:<20s}  {"YES" if val else "no"}')

        # ── Timestamp parsing ─────────────────────────────────────────────
        ts_cols = detect_ts_cols(df)
        for tc in ts_cols:
            section(f'Timestamp [{tc}]')
            numeric = pd.to_numeric(df[tc], errors='coerce')
            if numeric.notna().any():
                t_min, t_max = numeric.min(), numeric.max()
                unit = 's' if t_min > 1e9 else ('ms' if t_min > 1e12 else '?')
                print(f'    numeric  min: {t_min}  max: {t_max}  (likely {unit})')
                if unit == 's':
                    print(f'    → {pd.to_datetime(t_min, unit="s")}  to  {pd.to_datetime(t_max, unit="s")}')
                v2x_data[run.name][fpath.name]['ts_col']    = tc
                v2x_data[run.name][fpath.name]['ts_series'] = numeric.dropna()
                v2x_data[run.name][fpath.name]['t_min']     = t_min
                v2x_data[run.name][fpath.name]['t_max']     = t_max
            else:
                parsed = pd.to_datetime(df[tc], errors='coerce').dropna()
                if not parsed.empty:
                    print(f'    string  min: {parsed.min()}  max: {parsed.max()}')

---
## D. Radar JSON Files

Files like `Radars_Run_XXXX_sensor1.json` (per-sensor detections, ~1–2 MB each)  
and `Radars_Run_XXXX_traffic-triggers-output.json` (~6–9 MB).  
We inspect structure only — no heavy processing.

In [ ]:
# ── D1. Detect radar JSON files ────────────────────────────────────────────
RADAR_PATTERN   = [r'^Radars_Run_\d+_sensor\d+\.json$']
TRIGGER_PATTERN = [r'traffic.triggers.*\.json$']

print('Radar JSON files found:\n')
radar_index = {}

for run in selected_runs:
    sensor_files  = find_files(run, RADAR_PATTERN)
    trigger_files = find_files(run, TRIGGER_PATTERN)
    radar_index[run.name] = {'sensors': sensor_files, 'triggers': trigger_files}

    if sensor_files or trigger_files:
        for f in sensor_files:
            print(f'  {run.name:<12}  [sensor ] {f.name:50s}  {fmt_size(f)}')
        for f in trigger_files:
            print(f'  {run.name:<12}  [trigger] {f.name:50s}  {fmt_size(f)}')
    else:
        print(f'  {run.name:<12}  [none found]')

In [ ]:
# ── D2. Inspect one radar sensor JSON per run ──────────────────────────────
# We read the file in chunks and inspect structure only.

def safe_load_json(path: Path, max_bytes: int = 500_000) -> object:
    """Load a JSON file, truncating to max_bytes for safety."""
    size = path.stat().st_size
    if size > max_bytes:
        # Read partial bytes and try to parse as much as possible
        with open(path, 'r', encoding='utf-8', errors='ignore') as f:
            raw = f.read(max_bytes)
        # Full load anyway — json.load is fast for 2 MB
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        return json.load(f)


def describe_json(data, indent=4, max_items=3) -> str:
    """Pretty-print a compact sample of any JSON structure."""
    if isinstance(data, dict):
        sample = {k: data[k] for k in list(data.keys())[:max_items]}
    elif isinstance(data, list):
        sample = data[:max_items]
    else:
        sample = data
    return json.dumps(sample, indent=indent, default=str)[:2000]


radar_data = {}

for run in selected_runs:
    sensor_files = radar_index[run.name]['sensors']
    if not sensor_files:
        continue

    # Inspect only the first sensor file to avoid repetition
    fpath = sensor_files[0]
    run_header(run.name, f'— Radar Sensor JSON ({fpath.name})')
    print(f'  File size: {fmt_size(fpath)}')

    try:
        data = safe_load_json(fpath)
    except Exception as e:
        print(f'  [ERROR] Could not parse JSON: {e}')
        continue

    radar_data.setdefault(run.name, {})[fpath.name] = data

    section('Top-level type & structure')
    print(f'    type   : {type(data).__name__}')

    if isinstance(data, dict):
        print(f'    keys   : {list(data.keys())}')
        for k, v in data.items():
            print(f'    [{k}] → {type(v).__name__}', end='')
            if isinstance(v, list):   print(f'  len={len(v)}')
            elif isinstance(v, dict): print(f'  keys={list(v.keys())[:6]}')
            else:                     print(f'  = {str(v)[:80]}')

    elif isinstance(data, list):
        print(f'    length : {len(data)}')
        if data:
            first = data[0]
            print(f'    item[0] type: {type(first).__name__}')
            if isinstance(first, dict):
                print(f'    item[0] keys: {list(first.keys())}')

    section('Sample (first 3 records / top keys)')
    print(describe_json(data))

    # ── Dig one level deeper ──────────────────────────────────────────────
    # Find the main list of detection records
    records = None
    if isinstance(data, list):
        records = data
    elif isinstance(data, dict):
        for k, v in data.items():
            if isinstance(v, list) and len(v) > 0:
                records = v
                print(f'\n  Main record list found under key: "{k}"  ({len(v)} records)')
                break

    if records and isinstance(records[0], dict):
        section('One example record (full)')
        print(json.dumps(records[0], indent=4, default=str)[:2000])

        section('Nested keys in example record')
        for k, v in records[0].items():
            print(f'    {k:<30s}  {type(v).__name__}', end='')
            if isinstance(v, dict):   print(f'  keys={list(v.keys())[:6]}')
            elif isinstance(v, list): print(f'  len={len(v)}')
            else:                     print(f'  = {str(v)[:60]}')

        section('Content type inference for sensor JSON')
        all_keys = ' '.join(str(k) for r in records[:10] for k in (r.keys() if isinstance(r, dict) else []))
        inferences = {
            'detections/objects' : bool(re.search(r'detect|object|target|track|id', all_keys, re.I)),
            'position (x/y/lat/lon)' : bool(re.search(r'\bx\b|\by\b|lat|lon|pos|dist|range', all_keys, re.I)),
            'velocity/speed'     : bool(re.search(r'speed|vel|vx|vy|heading', all_keys, re.I)),
            'timestamps'         : bool(re.search(r'time|stamp|epoch|ts', all_keys, re.I)),
            'signal/traffic'     : bool(re.search(r'signal|phase|event|trig|state|traffic', all_keys, re.I)),
            'sensor metadata'    : bool(re.search(r'sensor|radar|device|channel|freq', all_keys, re.I)),
        }
        for k, v in inferences.items():
            print(f'    {k:<30s}  {"YES" if v else "no"}')

In [ ]:
# ── D3. Deep inspect: traffic-triggers-output.json ────────────────────────
# This file is likely the most important radar file — it may contain
# conflict triggers, signal phase events, or crossing detections.

for run in selected_runs:
    trigger_files = radar_index[run.name]['triggers']
    if not trigger_files:
        print(f'{run.name}: No traffic-triggers file found.')
        continue

    fpath = trigger_files[0]
    run_header(run.name, f'— Traffic Triggers JSON ({fpath.name})')
    print(f'  File size: {fmt_size(fpath)}')

    try:
        data = safe_load_json(fpath)
    except Exception as e:
        print(f'  [ERROR] Could not parse JSON: {e}')
        continue

    radar_data.setdefault(run.name, {})[fpath.name] = data

    section('Top-level structure')
    if isinstance(data, dict):
        print(f'    keys ({len(data)}): {list(data.keys())}')
        for k, v in data.items():
            print(f'    [{k}] → {type(v).__name__}', end='')
            if isinstance(v, list):   print(f'  len={len(v)}')
            elif isinstance(v, dict): print(f'  keys={list(v.keys())[:8]}')
            else:                     print(f'  = {str(v)[:80]}')
    elif isinstance(data, list):
        print(f'    list of {len(data)} items')
        if data:
            print(f'    item[0]: {type(data[0]).__name__}')

    # Flatten to a list of records
    records = data if isinstance(data, list) else None
    if isinstance(data, dict):
        for k, v in data.items():
            if isinstance(v, list) and len(v) > 0:
                records = v
                print(f'\n  Main record list under "{k}" — {len(v)} records')
                break

    if records is None:
        section('Full sample (no record list detected)')
        print(describe_json(data))
    else:
        section('Example record [0]')
        print(json.dumps(records[0], indent=4, default=str)[:3000])

        if len(records) > 1:
            section('Example record [1]  (for comparison)')
            print(json.dumps(records[1], indent=4, default=str)[:1500])

        section('All keys across first 20 records')
        all_keys_set = set()
        for r in records[:20]:
            if isinstance(r, dict):
                all_keys_set.update(r.keys())
        print(f'    {sorted(all_keys_set)}')

        section('Signal/conflict inference for triggers JSON')
        all_key_str = ' '.join(sorted(all_keys_set)).lower()
        # Also sample values for richer inference
        val_sample  = ' '.join(str(v) for r in records[:5] if isinstance(r, dict)
                               for v in r.values()).lower()
        combined    = all_key_str + ' ' + val_sample

        inferences = {
            'signal phases'      : bool(re.search(r'phase|green|red|yellow|amber|signal', combined)),
            'conflict triggers'  : bool(re.search(r'conflict|trigger|alert|warning|event', combined)),
            'traffic events'     : bool(re.search(r'event|traffic|approach|depart', combined)),
            'crossing/pedestrian': bool(re.search(r'cross|pedestrian|vru|ped|walk', combined)),
            'vehicle detections' : bool(re.search(r'vehicle|car|truck|motor|class', combined)),
            'timestamps present' : bool(re.search(r'time|stamp|epoch|ts\b', combined)),
            'zone/lane/approach' : bool(re.search(r'zone|lane|approach|entry|exit|arm', combined)),
        }
        for k, v in inferences.items():
            print(f'    {k:<30s}  {"✓ YES" if v else "✗ no"}')

        # ── Timestamp range in triggers ───────────────────────────────────
        ts_keys = [k for k in all_keys_set if re.search(r'time|stamp|epoch|ts\b', k, re.I)]
        if ts_keys:
            section(f'Timestamp field(s) in triggers: {ts_keys}')
            for ts_key in ts_keys:
                vals = [r.get(ts_key) for r in records if isinstance(r, dict) and ts_key in r]
                numeric_vals = pd.to_numeric(pd.Series(vals), errors='coerce').dropna()
                if not numeric_vals.empty:
                    t_min, t_max = numeric_vals.min(), numeric_vals.max()
                    unit = 's' if t_min > 1e9 else 'ms' if t_min > 1e12 else '?'
                    print(f'    [{ts_key}]  min: {t_min}  max: {t_max}  (unit: {unit})')
                    if unit == 's':
                        print(f'    → {pd.to_datetime(t_min, unit="s")}  to  {pd.to_datetime(t_max, unit="s")}')
                else:
                    print(f'    [{ts_key}]  non-numeric. Sample: {vals[:3]}')

---
## E. Cross-File Timestamp Alignment

Compare timestamp ranges across all file types for each run to understand time coverage and overlap.

In [ ]:
# ── E1. Load GT timestamps for comparison (already in notebook 00, re-load here)
GT_PATTERN = [r'^Run_\d+_GT\.csv$']
gt_ts_index = {}

for run in selected_runs:
    gt_files = find_files(run, GT_PATTERN)
    if not gt_files:
        gt_ts_index[run.name] = None
        continue
    df = safe_read_csv(gt_files[0], nrows=None)
    if df is None:
        gt_ts_index[run.name] = None
        continue
    ts_col = next((c for c in df.columns if re.search(r'\btime\b|\bts\b|timestamp', c, re.I)), None)
    if ts_col:
        ts = pd.to_numeric(df[ts_col], errors='coerce').dropna()
        gt_ts_index[run.name] = {
            'source': gt_files[0].name,
            'ts_col': ts_col,
            'fmt'   : 'Unix epoch (float s)',
            't_min' : ts.min(),
            't_max' : ts.max(),
            'n'     : len(ts),
        }
    else:
        gt_ts_index[run.name] = None

print('GT timestamps loaded for:', [r.name for r in selected_runs if gt_ts_index.get(r.name)])

In [ ]:
# ── E2. Build cross-file timestamp summary table per run ───────────────────

def unix_to_str(t):
    try:
        return str(pd.to_datetime(float(t), unit='s'))[:19]
    except Exception:
        return str(t)


for run in selected_runs:
    run_header(run.name, '— Timestamp Alignment')

    rows = []

    # ── GT ────────────────────────────────────────────────────────────────
    gt_info = gt_ts_index.get(run.name)
    if gt_info:
        rows.append({
            'Source'    : gt_info['source'],
            'TS Column' : gt_info['ts_col'],
            'Format'    : gt_info['fmt'],
            'Min time'  : unix_to_str(gt_info['t_min']),
            'Max time'  : unix_to_str(gt_info['t_max']),
            'N rows'    : gt_info['n'],
        })

    # ── ISC timing ────────────────────────────────────────────────────────
    for fname, info in isc_timing_data.get(run.name, {}).items():
        ts_s = info.get('ts_series')
        if ts_s is not None and not ts_s.empty:
            t_min, t_max = ts_s.min(), ts_s.max()
            fmt = 'Unix epoch (float s)' if t_min > 1e9 else 'numeric'
            rows.append({
                'Source'    : fname,
                'TS Column' : info.get('ts_col', '?'),
                'Format'    : fmt,
                'Min time'  : unix_to_str(t_min) if t_min > 1e9 else str(t_min),
                'Max time'  : unix_to_str(t_max) if t_max > 1e9 else str(t_max),
                'N rows'    : len(ts_s),
            })

    # ── Camera frame-timing (one camera) ─────────────────────────────────
    for kind in ['visual', 'thermal']:
        cam_info = cam_timing_data.get(run.name, {}).get(kind)
        if cam_info and 't_min' in cam_info:
            rows.append({
                'Source'    : cam_info['path'].name,
                'TS Column' : cam_info.get('ts_col', 'Timestamp'),
                'Format'    : 'ISC (YYYY-MM-DD-HH-MM-SS_us)',
                'Min time'  : unix_to_str(cam_info['t_min']),
                'Max time'  : unix_to_str(cam_info['t_max']),
                'N rows'    : cam_info.get('n_frames', '?'),
            })

    # ── V2X ───────────────────────────────────────────────────────────────
    for fname, info in v2x_data.get(run.name, {}).items():
        if 't_min' in info:
            t_min, t_max = info['t_min'], info['t_max']
            fmt = 'Unix epoch (s)' if t_min > 1e9 else 'numeric'
            rows.append({
                'Source'    : fname,
                'TS Column' : info.get('ts_col', '?'),
                'Format'    : fmt,
                'Min time'  : unix_to_str(t_min) if t_min > 1e9 else str(t_min),
                'Max time'  : unix_to_str(t_max) if t_max > 1e9 else str(t_max),
                'N rows'    : len(info.get('ts_series', [])),
            })

    # ── Radar triggers (if timestamp found in D3) ─────────────────────────
    # (Added automatically if inspect_triggers stored ts info in radar_data)

    if not rows:
        print('  No timestamp data available to compare.')
        continue

    align_df = pd.DataFrame(rows)
    print(align_df.to_string(index=False))

    # ── Overlap check ─────────────────────────────────────────────────────
    # Convert Min/Max time back to floats for comparison
    section('Overlap check (approximate)')
    unix_ranges = []
    for r in rows:
        try:
            t_min_u = pd.to_datetime(r['Min time']).timestamp()
            t_max_u = pd.to_datetime(r['Max time']).timestamp()
            unix_ranges.append((r['Source'], t_min_u, t_max_u))
        except Exception:
            pass

    if len(unix_ranges) >= 2:
        overall_min = min(t for _, t, _ in unix_ranges)
        overall_max = max(t for _, _, t in unix_ranges)
        print(f'  Overall span: {unix_to_str(overall_min)}  →  {unix_to_str(overall_max)}')
        print()
        for src, t_min, t_max in unix_ranges:
            # Timeline bar
            span = overall_max - overall_min if overall_max > overall_min else 1
            start_frac = (t_min - overall_min) / span
            end_frac   = (t_max - overall_min) / span
            bar_width  = 40
            bar = ' ' * int(start_frac * bar_width) + '█' * max(1, int((end_frac - start_frac) * bar_width))
            bar = bar.ljust(bar_width)
            print(f'  {src[:40]:<40s}  |{bar}|')

In [ ]:
# ── E3. Timeline visualisation for the first selected run ─────────────────
# Show all timestamp ranges on a common axis as horizontal bars.

first_run = selected_runs[0]

viz_rows = []

gt_info = gt_ts_index.get(first_run.name)
if gt_info:
    viz_rows.append(('GT file', gt_info['t_min'], gt_info['t_max']))

for fname, info in isc_timing_data.get(first_run.name, {}).items():
    ts_s = info.get('ts_series')
    if ts_s is not None and not ts_s.empty:
        viz_rows.append((fname[:35], ts_s.min(), ts_s.max()))

for kind in ['visual', 'thermal']:
    cam_info = cam_timing_data.get(first_run.name, {}).get(kind)
    if cam_info and 't_min' in cam_info:
        viz_rows.append((cam_info['path'].name[:35], cam_info['t_min'], cam_info['t_max']))

for fname, info in v2x_data.get(first_run.name, {}).items():
    if 't_min' in info:
        viz_rows.append((fname[:35], info['t_min'], info['t_max']))

if viz_rows:
    labels  = [r[0] for r in viz_rows]
    t_mins  = [r[1] for r in viz_rows]
    t_maxes = [r[2] for r in viz_rows]

    # Normalise to seconds-from-start for readability
    t_origin = min(t_mins)
    t_mins_n  = [t - t_origin for t in t_mins]
    t_maxes_n = [t - t_origin for t in t_maxes]
    widths    = [mx - mn for mn, mx in zip(t_mins_n, t_maxes_n)]

    fig, ax = plt.subplots(figsize=(11, max(3, len(labels) * 0.55)))
    colors  = plt.cm.tab10.colors
    yticks  = list(range(len(labels)))

    for i, (lbl, start, width) in enumerate(zip(labels, t_mins_n, widths)):
        ax.barh(i, width, left=start, height=0.55,
                color=colors[i % len(colors)], edgecolor='white', linewidth=0.5)

    ax.set_yticks(yticks)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel('Elapsed time (s) from earliest timestamp')
    ax.set_title(f'Timestamp Range per File — {first_run.name}', fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.show()
else:
    print('No timestamp ranges available for timeline plot.')

---
## F. Human-Readable Summary Table

A single reference table covering every non-GT file type.

In [ ]:
# ── F. Summary table ──────────────────────────────────────────────────────
summary = [
    {
        'File pattern'              : 'ISC_Run_XXXX_ISC_all_timing.csv',
        'Likely purpose'            : 'Cross-sensor synchronisation metadata for the run',
        'Useful now?'               : 'Maybe',
        'Contains timestamps?'      : 'Yes',
        'Object-level data?'        : 'No',
        'Signal-related info?'      : 'Possibly (sensor sync offsets)',
        'Notes'                     : 'Small file (~2 KB). Check columns for offset/delay fields.',
    },
    {
        'File pattern'              : 'ISC_Run_XXXX_v2xhub_timing.csv',
        'Likely purpose'            : 'V2X hub heartbeat / connectivity timing log',
        'Useful now?'               : 'Later',
        'Contains timestamps?'      : 'Yes',
        'Object-level data?'        : 'No',
        'Signal-related info?'      : 'Maybe (hub message timing)',
        'Notes'                     : 'Very small (~276 B). May be empty or minimal.',
    },
    {
        'File pattern'              : 'VisualCameraX_Run_XXXX_frame-timing.csv',
        'Likely purpose'            : 'Frame index ↔ wall-clock timestamp for visual camera X',
        'Useful now?'               : 'Yes',
        'Contains timestamps?'      : 'Yes (ISC string format)',
        'Object-level data?'        : 'No',
        'Signal-related info?'      : 'No',
        'Notes'                     : 'Links Image_number → timestamp. ~1450 frames/file at ~10 fps.',
    },
    {
        'File pattern'              : 'ThermalCameraX_Run_XXXX_frame-timing.csv',
        'Likely purpose'            : 'Frame index ↔ wall-clock timestamp for thermal camera X',
        'Useful now?'               : 'Yes',
        'Contains timestamps?'      : 'Yes (ISC string format)',
        'Object-level data?'        : 'No',
        'Signal-related info?'      : 'No',
        'Notes'                     : 'Same schema as visual timing. 5 cameras per run.',
    },
    {
        'File pattern'              : 'V2XHubSensor_Run_XXXX.csv',
        'Likely purpose'            : 'V2X hub sensor log (SPaT / BSM / MAP messages)',
        'Useful now?'               : 'Yes',
        'Contains timestamps?'      : 'Yes',
        'Object-level data?'        : 'Maybe (BSM = vehicle beacons)',
        'Signal-related info?'      : 'Yes — likely contains SPaT (signal phase & timing)',
        'Notes'                     : 'Only present in some runs. Key file for signal info.',
    },
    {
        'File pattern'              : 'Radars_Run_XXXX_sensorN.json',
        'Likely purpose'            : 'Per-sensor radar detections (objects, positions, velocities)',
        'Useful now?'               : 'Maybe',
        'Contains timestamps?'      : 'Yes (likely)',
        'Object-level data?'        : 'Yes',
        'Signal-related info?'      : 'No',
        'Notes'                     : '1–2 MB per sensor. 4 sensors per run. Inspect keys carefully.',
    },
    {
        'File pattern'              : 'Radars_Run_XXXX_traffic-triggers-output.json',
        'Likely purpose'            : 'Processed radar events: zone crossings, triggers, conflict alerts',
        'Useful now?'               : 'Yes',
        'Contains timestamps?'      : 'Yes (likely)',
        'Object-level data?'        : 'Yes',
        'Signal-related info?'      : 'Yes — may contain conflict/phase trigger events',
        'Notes'                     : '6–9 MB. Most informative radar file. Deep inspect recommended.',
    },
    {
        'File pattern'              : 'Run_XXXX_GT.csv  (reference)',
        'Likely purpose'            : 'Ground truth object tracks per frame',
        'Useful now?'               : 'Yes (already inspected)',
        'Contains timestamps?'      : 'Yes (Unix float, ~10 Hz)',
        'Object-level data?'        : 'Yes',
        'Signal-related info?'      : 'No',
        'Notes'                     : 'Covered in notebook 00. Use as alignment reference.',
    },
]

summary_df = pd.DataFrame(summary)

# Pretty print
pd.set_option('display.max_colwidth', 60)
print('Non-GT File Summary\n')
print(summary_df.to_string(index=False))

# ── Highlight what to prioritise for EDA ──────────────────────────────────
print('\n' + '='*65)
print('  FILES TO PRIORITISE FOR EDA')
print('='*65)
priority = summary_df[summary_df['Useful now?'] == 'Yes']
for _, row in priority.iterrows():
    print(f'  ✓ {row["File pattern"]}')
    print(f'    → {row["Likely purpose"]}')
    if row['Signal-related info?'] not in ('No', 'no'):
        print(f'    ⚡ Signal info: {row["Signal-related info?"]}')
    print()